In [1]:
%pip install pandas requests --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import io, requests
import pandas as pd

INPUT_PATH = "locations-raw.csv"      # change if needed
OUTPUT_PATH = "locations-geocoded.csv"

df = pd.read_csv(INPUT_PATH).reset_index(drop=True)

In [3]:
# Build the headerless batch payload: id, street, city, state, zip
batch = pd.DataFrame({
    "id": df.index,
    "street": df["Site Address"].str.strip(),
    "city": "San Francisco",
    "state": "CA",
    "zip": "",
})
csv_buf = io.StringIO()
batch.to_csv(csv_buf, header=False, index=False)

In [4]:
# One request geocodes all rows (free, no key, US-only)
resp = requests.post(
    "https://geocoding.geo.census.gov/geocoder/locations/addressbatch",
    files={"addressFile": ("addresses.csv", csv_buf.getvalue())},
    data={"benchmark": "Public_AR_Current"},
    timeout=180,
)
resp.raise_for_status()

In [5]:
cols = ["id", "input_address", "match", "match_type",
        "matched_address", "coordinates", "tiger_line_id", "side"]
geo = pd.read_csv(io.StringIO(resp.text), header=None, names=cols,
                  dtype={"id": int}).sort_values("id").reset_index(drop=True)

coords = geo["coordinates"].str.split(",", expand=True)
geo["longitude"] = pd.to_numeric(coords[0], errors="coerce")
geo["latitude"]  = pd.to_numeric(coords[1], errors="coerce")

result = df.join(geo.set_index("id")[["match", "matched_address", "longitude", "latitude"]])
result["zip"] = result["matched_address"].str.extract(r"(\d{5})(?:-\d{4})?\s*$")

In [6]:
# manually deal with rows with NaN
manual_fixes = {
    "1485 Bay Shore Blvd": "1485 Bayshore Blvd, San Francisco, CA 94124",
    "5th St & Brannan St": "5th St & Brannan St, San Francisco, CA 94107",
    "1201 08th St":        "1201 8th St, San Francisco, CA 94107",
}

for site, full in manual_fixes.items():
    mask = result["Site Address"] == site
    result.loc[mask, "matched_address"] = full
    result.loc[mask, "zip"] = pd.Series([full]).str.extract(r"(\d{5})(?:-\d{4})?\s*$").iloc[0, 0]
    result.loc[mask, "match"] = "Match (manual)"

result[result["Site Address"].isin(manual_fixes)][["Site Address", "matched_address", "zip", "match"]]

,Site Address,matched_address,zip,match
7,1485 Bay Shore Blvd,"1485 Bayshore Blvd, San Francisco, CA 94124",94124,Match (manual)
8,5th St & Brannan St,"5th St & Brannan St, San Francisco, CA 94107",94107,Match (manual)
16,1201 08th St,"1201 8th St, San Francisco, CA 94107",94107,Match (manual)


In [7]:
# Geocode the 3 fixed addresses to fill in coordinates (Census single-line, keyless)
def geocode_oneline(address):
    r = requests.get(
        "https://geocoding.geo.census.gov/geocoder/locations/onelineaddress",
        params={"address": address, "benchmark": "Public_AR_Current", "format": "json"},
        timeout=60,
    )
    r.raise_for_status()
    matches = r.json()["result"]["addressMatches"]
    if matches:
        c = matches[0]["coordinates"]
        return c["y"], c["x"]          # (latitude, longitude)
    return None, None

for site, full in manual_fixes.items():
    lat, lon = geocode_oneline(full)
    mask = result["Site Address"] == site
    result.loc[mask, "latitude"] = lat
    result.loc[mask, "longitude"] = lon
    print(f"{site:<22} -> {lat}, {lon}")

# add coordinate for the intersection (Census doesn't geocode intersections)
mask = result["Site Address"] == "5th St & Brannan St"
result.loc[mask, ["latitude", "longitude"]] = [37.776634, -122.398814]

result[result["Site Address"].isin(manual_fixes)][["Site Address", "matched_address", "latitude", "longitude"]]

1485 Bay Shore Blvd    -> 37.725535736384, -122.401401989518
5th St & Brannan St    -> None, None
1201 08th St           -> 37.766696846569, -122.399573812702


,Site Address,matched_address,latitude,longitude
7,1485 Bay Shore Blvd,"1485 Bayshore Blvd, San Francisco, CA 94124",37.725536,-122.401402
8,5th St & Brannan St,"5th St & Brannan St, San Francisco, CA 94107",37.776634,-122.398814
16,1201 08th St,"1201 8th St, San Francisco, CA 94107",37.766697,-122.399574


In [8]:
# Rows that still did not match (fix source address or geocode manually)
result[result["match"] != "Match"][["Site Address", "match"]]

,Site Address,match
7,1485 Bay Shore Blvd,Match (manual)
8,5th St & Brannan St,Match (manual)
16,1201 08th St,Match (manual)


In [9]:
# drop the match column, then save
final = result.drop(columns=["match"])
final.to_csv(OUTPUT_PATH, index=False)
final[["Site Address", "matched_address", "zip", "latitude", "longitude"]]

,Site Address,matched_address,zip,latitude,longitude
0,50 Quint St,"50 QUINT ST, SAN FRANCISCO, CA, 94124",94124,37.746304,-122.388371
1,1111 Pennsylvania Ave,"1111 PENNSYLVANIA AVE, SAN FRANCISCO, CA, 94107",94107,37.752470,-122.392558
2,330 8th St,"330 8TH ST, SAN FRANCISCO, CA, 94103",94103,37.774802,-122.409928
3,1101-1123 Sutter St,"1123 SUTTER ST, SAN FRANCISCO, CA, 94109",94109,37.787854,-122.418863
4,2450 Alameda St,"2450 ALAMEDA ST, SAN FRANCISCO, CA, 94103",94103,37.768327,-122.409236
5,1200 Larkin St,"1200 LARKIN ST, SAN FRANCISCO, CA, 94109",94109,37.789440,-122.418691
6,1500 19th Ave,"1500 19TH AVE, SAN FRANCISCO, CA, 94122",94122,37.759784,-122.476683
7,1485 Bay Shore Blvd,"1485 Bayshore Blvd, San Francisco, CA 94124",94124,37.725536,-122.401402
8,5th St & Brannan St,"5th St & Brannan St, San Francisco, CA 94107",94107,37.776634,-122.398814
9,1160 Mission St,"1160 MISSION ST, SAN FRANCISCO, CA, 94103",94103,37.778319,-122.412173
